In [1]:
import pandas as pd
import numpy as np

sp = pd.read_csv("../data/dailytotalreturn_S&P.csv")

sp.head()
sp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8817 entries, 0 to 8816
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   INDNO      8817 non-null   int64  
 1   DlyCalDt   8817 non-null   object 
 2   DlyTotRet  8817 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 206.8+ KB


In [2]:
from scipy.stats import jarque_bera, anderson

r = sp["DlyTotRet"].values

def sample_tests(m=1000, reps=200, seed=0):
    rng = np.random.default_rng(seed)
    jb_pvals = []
    ad_stats = []
    for _ in range(reps):
        samp = rng.choice(r, size=m, replace=False)
        jb_pvals.append(jarque_bera(samp)[1])
        ad_stats.append(anderson(samp, dist="norm").statistic)
    return np.array(jb_pvals), np.array(ad_stats)

jb_pvals_1000, ad_stats_1000 = sample_tests(m=1000, reps=200, seed=1)

print("JB: quota p<0.05 con m=1000:", np.mean(jb_pvals_1000 < 0.05))
print("AD statistic (mediana) con m=1000:", np.median(ad_stats_1000))

JB: quota p<0.05 con m=1000: 1.0
AD statistic (mediana) con m=1000: 16.918750821817127


In [3]:
import numpy as np
from scipy.stats import norm

r = sp["DlyTotRet"]
mu, sigma = r.mean(), r.std()

for k in [3,4,5]:
    p_real = np.mean(np.abs(r-mu) > k*sigma)
    p_norm = 2*(1-norm.cdf(k))
    print(f">{k}σ: reale={p_real:.6f}, normale={p_norm:.6f}, moltiplicatore={p_real/p_norm:.1f}x")

>3σ: reale=0.015538, normale=0.002700, moltiplicatore=5.8x
>4σ: reale=0.006351, normale=0.000063, moltiplicatore=100.3x
>5σ: reale=0.003289, normale=0.000001, moltiplicatore=5737.1x
